# Bronze Layer: Wind Turbine Raw Data Ingest

Ingests raw CSV files from the wind turbine landing zone into a bronze Delta table using Databricks Autoloader.

## Design
- **Source**: Three CSV files (`data_group_1.csv`, `data_group_2.csv`, `data_group_3.csv`), each containing readings for 5 turbines (15 total).
- **Pattern**: Autoloader (`cloudFiles`) streams new/updated files incrementally; checkpoint state prevents reprocessing.
- **Output**: A single `bronze.wind_turbine_raw` Delta table with all raw rows plus ingest metadata columns.
- **Schema handling**: Explicit schema is enforced at ingest; `cloudFiles.schemaEvolutionMode = rescue` captures any unexpected columns into `_rescued_data` rather than failing the stream.

## Assumptions
- CSV columns: `timestamp`, `turbine_id`, `wind_speed_ms`, `wind_direction_deg`, `power_output_mw`.
- Files land under a single root path, organised as `<landing_zone>/data_group_*.csv`.
- Autoloader runs in triggered (once) mode so it can be scheduled as a daily job task.

In [ ]:
# ---------------------------------------------------------------------------
# Parameters (overridden by job/widget at runtime)
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog",          "wind_farm_dev")
dbutils.widgets.text("schema",           "wind_farm")
dbutils.widgets.text("landing_zone",     "/Volumes/wind_farm_dev/wind_farm/landing")
dbutils.widgets.text("checkpoint_base",  "/Volumes/wind_farm_dev/wind_farm/checkpoints/bronze")

catalog         = dbutils.widgets.get("catalog")
schema          = dbutils.widgets.get("schema")
landing_zone    = dbutils.widgets.get("landing_zone")
checkpoint_base = dbutils.widgets.get("checkpoint_base")

print(catalog)
print(schema)

bronze_table    = f"`{catalog}`.`{schema}`.wind_turbine_raw"
checkpoint_path = f"{checkpoint_base}/wind_turbine_raw"
schema_path     = f"{checkpoint_base}/wind_turbine_raw_schema"

print(f"Catalog       : {catalog}")
print(f"Schema        : {schema}")
print(f"Target table  : {bronze_table}")
print(f"Landing zone  : {landing_zone}")
print(f"Checkpoint    : {checkpoint_path}")

In [ ]:
from pyspark.sql import types as T

# Explicit schema avoids costly full-scan inference on each run.
# Any column not listed here is captured in _rescued_data.
TURBINE_SCHEMA = T.StructType([
    T.StructField("timestamp",          T.TimestampType(), nullable=True),
    T.StructField("turbine_id",         T.StringType(),    nullable=False),
    T.StructField("wind_speed",      T.DoubleType(),    nullable=True),
    T.StructField("wind_direction", T.DoubleType(),    nullable=True),
    T.StructField("power_output",    T.DoubleType(),    nullable=True),
])

In [ ]:
from pyspark.sql import functions as F

raw_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",             "csv")
        .option("header",                         "true")
        .schema(TURBINE_SCHEMA)
        .option("cloudFiles.schemaLocation",     schema_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.useIncrementalListing", "true")
        .load(landing_zone)
        .withColumn("_source_file",         F.col("_metadata.file_path"))
        .withColumn("_source_modified_at",  F.col("_metadata.file_modification_time"))
        .withColumn("_ingested_at",         F.current_timestamp())
)

In [ ]:
# ---------------------------------------------------------------------------
# Ensure target catalog / schema exist before writing
# ---------------------------------------------------------------------------
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

In [ ]:
# ---------------------------------------------------------------------------
# Write bronze Delta table
# ---------------------------------------------------------------------------
# availableNow = process all files that have arrived since last checkpoint,
# then stop — suitable for a scheduled daily job.
(
    raw_stream.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .option("mergeSchema",        "true")
        .trigger(availableNow=True)
        .toTable(bronze_table)
        .awaitTermination()
)

print(f"Bronze ingest complete → {bronze_table}")

In [ ]:
# ---------------------------------------------------------------------------
# Quick sanity check
# ---------------------------------------------------------------------------
bronze_df = spark.table(bronze_table)

print(f"Total rows ingested : {bronze_df.count():,}")
print(f"Distinct turbines   : {bronze_df.select('turbine_id').distinct().count()}")
print(f"Date range          : {bronze_df.agg(F.min('timestamp'), F.max('timestamp')).collect()[0]}")
print("\nSource files processed:")
bronze_df.groupBy("_source_file").count().orderBy("_source_file").show(truncate=False)